# Video tracking and MOT export

Run SAHI sliced detection and ByteTrack in one pass. The workflow writes an annotated MP4 and a matching MOTChallenge-format prediction file:

```text
frame_id,track_id,x,y,width,height,confidence,-1,-1,-1
```

Set the paths and inference options below, then run the notebook from top to bottom.

In [2]:
from pathlib import Path
import re

import cv2
import numpy as np
import supervision as sv
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

/home/stiro/anaconda3/envs/image-object-detection/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_PATH = Path("models/best (3).pt")
INPUT_VIDEO = Path("videos/walk_people.mp4")
OUTPUT_DIR = Path("output")
OUTPUT_FILENAME = "tracked.mp4"

DEVICE = "cuda:0"
SLICE_SIZE = 640
OVERLAP = 0.1
CONFIDENCE = 0.2

## Optional: build a video from an image sequence

This helper is useful for VisDrone frame directories. Set `gaussian_blur=True` to create a degraded-input experiment without modifying the source frames.

In [9]:
import re
from pathlib import Path
import cv2


def _natural_key(path):
    return [
        (0, int(part)) if part.isdigit() else (1, part.casefold())
        for part in re.split(r"(\d+)", path.name)
    ]


def image_sequence_to_video(
    image_dir,
    output_path,
    *,
    fps=15.0,
    gaussian_blur=False,
):
    image_dir = Path(image_dir)
    output_path = Path(output_path)

    image_paths = sorted(image_dir.glob("*.jpg"), key=_natural_key)
    
    if not image_paths:
        raise ValueError(f"No JPG frames found in {image_dir}")

    first_frame = cv2.imread(str(image_paths[0]))
    if first_frame is None:
        raise ValueError(f"Cannot read {image_paths[0]}")

    height, width = first_frame.shape[:2]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not create {output_path}")

    try:
        for image_path in image_paths:
            frame = cv2.imread(str(image_path))
            if frame is None:
                raise ValueError(f"Cannot read {image_path}")
            if frame.shape[:2] != (height, width):
                frame = cv2.resize(frame, (width, height))
            if gaussian_blur:
                frame = cv2.GaussianBlur(
                    frame,
                    ksize=(7, 7),
                    sigmaX=1.5,
                    sigmaY=1.5,
                )
            writer.write(frame)
    finally:
        writer.release()

    return str(output_path)

In [10]:
sequence_path = Path("./videos/visdrone/sequences/uav0000086_00000_v")
save_path = Path("./videos/basketball.mp4")
image_sequence_to_video(sequence_path, save_path)

'videos/basketball.mp4'

## Track and export

Empty frames are still passed to ByteTrack so tracks expire correctly. Prediction frame IDs start at 1 and advance once per decoded video frame.

In [ ]:
def track_video(
    model_path,
    input_video,
    output_filename,
    *,
    output_dir="output",
    predictions_filename=None,
    slice_size=640,
    overlap=0.1,
    confidence=0.2,
    device="cuda:0",
):
    """Track a video and save annotated frames plus MOT-format predictions."""
    model_path = Path(model_path)
    input_video = Path(input_video)
    output_dir = Path(output_dir)

    if not model_path.is_file():
        raise FileNotFoundError(f"Model checkpoint not found: {model_path}")
    if not input_video.is_file():
        raise FileNotFoundError(f"Input video not found: {input_video}")
    if slice_size <= 0:
        raise ValueError("slice_size must be positive")
    if not 0 <= overlap < 1:
        raise ValueError("overlap must be in [0, 1)")
    if not 0 <= confidence <= 1:
        raise ValueError("confidence must be in [0, 1]")

    output_path = output_dir / output_filename
    predictions_path = (
        output_dir / predictions_filename
        if predictions_filename is not None
        else output_path.with_suffix(".txt")
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    predictions_path.parent.mkdir(parents=True, exist_ok=True)

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=str(model_path),
        confidence_threshold=confidence,
        device=device,
    )

    capture = cv2.VideoCapture(str(input_video))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {input_video}")

    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = capture.get(cv2.CAP_PROP_FPS)
    if width <= 0 or height <= 0 or fps <= 0:
        capture.release()
        raise ValueError("Input video has invalid dimensions or frame rate")

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f"Could not create {output_path}")

    tracker = sv.ByteTrack(frame_rate=round(fps))
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    trace_annotator = sv.TraceAnnotator(trace_length=30)

    frame_id = 0
    exported_detections = 0

    try:
        with predictions_path.open("w", encoding="utf-8") as predictions_file:
            while True:
                success, frame = capture.read()
                if not success:
                    break

                frame_id += 1
                result = get_sliced_prediction(
                    frame,
                    detection_model,
                    slice_height=slice_size,
                    slice_width=slice_size,
                    overlap_height_ratio=overlap,
                    overlap_width_ratio=overlap,
                    perform_standard_pred=False,
                    postprocess_type="GREEDYNMM",
                    postprocess_match_metric="IOS",
                    postprocess_match_threshold=0.5,
                    verbose=0,
                )

                predictions = result.object_prediction_list
                if predictions:
                    detections = sv.Detections(
                        xyxy=np.asarray(
                            [prediction.bbox.to_xyxy() for prediction in predictions],
                            dtype=np.float32,
                        ),
                        confidence=np.asarray(
                            [prediction.score.value for prediction in predictions],
                            dtype=np.float32,
                        ),
                        class_id=np.asarray(
                            [prediction.category.id for prediction in predictions],
                            dtype=int,
                        ),
                    )
                else:
                    detections = sv.Detections.empty()

                detections = tracker.update_with_detections(detections)

                if len(detections):
                    labels = []
                    for box, track_id, class_id, score in zip(
                        detections.xyxy,
                        detections.tracker_id,
                        detections.class_id,
                        detections.confidence,
                    ):
                        x1, y1, x2, y2 = box
                        predictions_file.write(
                            f"{frame_id},{int(track_id)},{x1:.3f},{y1:.3f},"
                            f"{x2 - x1:.3f},{y2 - y1:.3f},"
                            f"{score:.6f},-1,-1,-1\n"
                        )
                        labels.append(
                            f"#{int(track_id)} class={int(class_id)} {score:.2f}"
                        )
                        exported_detections += 1

                    frame = trace_annotator.annotate(
                        scene=frame,
                        detections=detections,
                    )
                    frame = box_annotator.annotate(
                        scene=frame,
                        detections=detections,
                    )
                    frame = label_annotator.annotate(
                        scene=frame,
                        detections=detections,
                        labels=labels,
                    )

                writer.write(frame)
    finally:
        capture.release()
        writer.release()

    return {
        "video": str(output_path),
        "predictions": str(predictions_path),
        "frames": frame_id,
        "detections": exported_detections,
    }

In [ ]:
MODEL_PATH = Path("models/model.pt")
OUTPUT_DIR = Path("output")
input_videos = [v for v in list(Path("videos/").iterdir()) if str(v).endswith(".mp4")]

DEVICE = "cuda:0"
SLICE_SIZE = 640
OVERLAP = 0.1
CONFIDENCE = 0.2

In [ ]:
for input_video in input_videos:
  output_filename = "tracking_" + input_video.name
  outputs = track_video(
      model_path=MODEL_PATH,
      input_video=input_video,
      output_filename=output_filename,
      output_dir=OUTPUT_DIR,
      slice_size=SLICE_SIZE,
      overlap=OVERLAP,
      confidence=CONFIDENCE,
      device=DEVICE,
  )
